# PatchTST Re-implementation Results
Reproduces key results from "A Time Series is Worth 64 Words" (Nie et al., ICLR 2023).

This notebook:
1. Runs experiments across ETT datasets and prediction horizons
2. Compares our MSE/MAE results against Table 3 of the paper
3. Generates figures for the poster and final report

**Before running:** Runtime → Change runtime type → T4 GPU

## 0. Setup

In [ ]:
import os

GITHUB_URL = "https://github.com/mmichellezhou/patchtst.git"

if os.path.exists("/content/patchtst"):
    !git -C /content/patchtst pull
else:
    !git clone {GITHUB_URL} /content/patchtst

if not os.path.exists("/content/patchtst/data/ETDataset/ETT-small/ETTh1.csv"):
    !rm -rf /content/patchtst/data/ETDataset
    !git clone https://github.com/zhouhaoyi/ETDataset.git /content/patchtst/data/ETDataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys, os

PROJ_ROOT   = "/content/patchtst"
CODE_DIR    = f"{PROJ_ROOT}/code"
RESULTS_DIR = "/content/drive/MyDrive/final project/results"
FIGURES_DIR = f"{RESULTS_DIR}/figures"
TABLES_DIR  = f"{RESULTS_DIR}/tables"
CKPT_DIR    = f"{RESULTS_DIR}/checkpoints"

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

for d in [FIGURES_DIR, TABLES_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Code:",    CODE_DIR)
print("Results:", RESULTS_DIR)


In [ ]:
!pip install -q -r "/content/patchtst/requirements.txt"


In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from config import Config
from train import train
from evaluate import load_and_evaluate

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Paper Reference Results (Table 3 — PatchTST/42, ETTh1)

In [ ]:
PAPER_RESULTS = {
    96:  {"MSE": 0.375, "MAE": 0.399},
    192: {"MSE": 0.414, "MAE": 0.421},
    336: {"MSE": 0.431, "MAE": 0.436},
    720: {"MSE": 0.449, "MAE": 0.466},
}

df_paper = pd.DataFrame(PAPER_RESULTS).T
df_paper.index.name = "pred_len"
print("Paper reported results (PatchTST/42, ETTh1):")
df_paper

## 2. Train and Evaluate Across Prediction Horizons

In [ ]:
PRED_LENS = [96, 192, 336, 720]

our_results    = {}
loss_histories = {}

for pred_len in PRED_LENS:
    print(f"\n{'='*50}")
    print(f"Training  pred_len = {pred_len}")
    print(f"{'='*50}")

    config = Config(dataset_name="ETTh1", pred_len=pred_len)
    config.data_path       = f"{PROJ_ROOT}/data/ETDataset/ETT-small/{config.dataset_name}.csv"
    config.save_path       = f"{RESULTS_DIR}/{pred_len}"
    config.checkpoint_path = f"{CKPT_DIR}/{pred_len}"
    os.makedirs(config.checkpoint_path, exist_ok=True)

    train_losses, val_losses = train(config)
    loss_histories[pred_len] = {"train": train_losses, "val": val_losses}

    mse, mae = load_and_evaluate(config, device=DEVICE)
    our_results[pred_len] = {"MSE": round(mse, 3), "MAE": round(mae, 3)}
    print(f"Test  MSE={mse:.3f}  MAE={mae:.3f}")

## 3. Results Comparison vs. Table 3

In [ ]:
df_ours = pd.DataFrame(our_results).T
df_ours.index.name = "pred_len"

df_compare = pd.concat(
    {"Ours": df_ours, "Paper (PatchTST/42)": df_paper},
    axis=1
)
print("ETTh1 — Ours vs. Paper:")
display(df_compare)

csv_path = f"{TABLES_DIR}/ETTh1_results.csv"
df_compare.to_csv(csv_path)
print(f"Saved to {csv_path}")

In [ ]:
pred_lens = list(PRED_LENS)
x = range(len(pred_lens))
width = 0.35

our_mse   = [our_results[t]["MSE"]   for t in pred_lens]
paper_mse = [PAPER_RESULTS[t]["MSE"] for t in pred_lens]
our_mae   = [our_results[t]["MAE"]   for t in pred_lens]
paper_mae = [PAPER_RESULTS[t]["MAE"] for t in pred_lens]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, our_vals, paper_vals, metric in zip(
    axes,
    [our_mse, our_mae],
    [paper_mse, paper_mae],
    ["MSE", "MAE"]
):
    ax.bar([i - width/2 for i in x], our_vals,   width, label="Ours",  color="steelblue")
    ax.bar([i + width/2 for i in x], paper_vals, width, label="Paper", color="tomato", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(pred_lens)
    ax.set_xlabel("Prediction Horizon")
    ax.set_ylabel(metric)
    ax.set_title(f"ETTh1 {metric} — Ours vs. Paper")
    ax.legend()

fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/ETTh1_comparison.png", dpi=150)
plt.show()

## 4. Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, pred_len in zip(axes.flatten(), PRED_LENS):
    h = loss_histories[pred_len]
    ax.plot(h["train"], label="train MSE")
    ax.plot(h["val"],   label="val MSE")
    ax.set_title(f"pred_len = {pred_len}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.legend()

fig.suptitle("PatchTST Training Loss — ETTh1", fontsize=13)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/ETTh1_loss_curves.png", dpi=150)
plt.show()

## 5. Forecast Visualization

In [ ]:
import numpy as np
from patchtst import PatchTST
from dataset import get_dataloaders

VIZ_PRED_LEN = 96

config = Config(dataset_name="ETTh1", pred_len=VIZ_PRED_LEN)
config.data_path       = f"{PROJ_ROOT}/data/ETDataset/ETT-small/{config.dataset_name}.csv"
config.checkpoint_path = f"{CKPT_DIR}/{VIZ_PRED_LEN}"

_, _, test_loader = get_dataloaders(config)
model = PatchTST(config).to(DEVICE)
model.load_state_dict(torch.load(f"{config.checkpoint_path}/best_model.pt", map_location=DEVICE))
model.eval()

x_batch, y_batch = next(iter(test_loader))
with torch.no_grad():
    pred_batch = model(x_batch.to(DEVICE)).cpu()

CHANNEL_NAMES = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]
n_channels = x_batch.shape[-1]
seq_len, pred_len = x_batch.shape[1], y_batch.shape[1]
t_in  = np.arange(seq_len)
t_out = np.arange(seq_len, seq_len + pred_len)

fig, axes = plt.subplots(n_channels, 1, figsize=(12, 3 * n_channels))
for i, ax in enumerate(axes):
    ax.plot(t_in,  x_batch[0, :, i].numpy(),    color="steelblue",               label="input")
    ax.plot(t_out, y_batch[0, :, i].numpy(),    color="steelblue", linestyle="--", label="ground truth")
    ax.plot(t_out, pred_batch[0, :, i].numpy(), color="tomato",                   label="prediction")
    ax.axvline(seq_len, color="gray", linestyle=":")
    ax.set_ylabel(CHANNEL_NAMES[i])
    if i == 0:
        ax.legend(loc="upper right")

axes[-1].set_xlabel("Time step")
fig.suptitle(f"PatchTST Forecast — ETTh1, pred_len={VIZ_PRED_LEN}", fontsize=13)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/ETTh1_forecast_T{VIZ_PRED_LEN}.png", dpi=150)
plt.show()